# 🔍 Phase 5: Explainable AI (XAI) using SHAP
### **SecureBank AI — Hybrid Fraud Intelligence System**

#### **Module Objectives:**
1. **Model Interpretability**: Demystify the "black-box" Random Forest classifier using Shapley Additive exPlanations (SHAP) grounded in cooperative game theory.
2. **Global Feature Attribution**: Identify the macro-drivers across the entire credit card portfolio (e.g. `V14`, `V10`, `V12`, `V17`, `V4`).
3. **Local Transaction Explanations**: Calculate exact Shapley value contributions for individual transaction investigations.
4. **Operational Audit Trail**: Provide defensible, regulatory-compliant explanations for Suspicious Activity Reports (SAR).
5. **Artifact Generation**: Export SHAP summary and waterfall plots for the interactive Streamlit dashboard.

In [ ]:
import os
import joblib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)
os.makedirs("../outputs/figures", exist_ok=True)
print(f"SHAP version: {shap.__version__}")

## 1. Load Pre-Trained Model & Test Data
We load the fitted `random_forest.pkl`, `scaler.pkl`, and test evaluation records.

In [ ]:
model_path = "../models/random_forest.pkl"
if not os.path.exists(model_path):
    model_path = "models/random_forest.pkl"

model = joblib.load(model_path)

test_path = "../data/processed/test_processed.csv"
if not os.path.exists(test_path):
    test_path = "data/processed/test_processed.csv"

test_df = pd.read_csv(test_path)
X_test = test_df.drop(columns=["Class"])
y_test = test_df["Class"]

print(f"Loaded model: {type(model).__name__}")
print(f"Total test instances available: {len(X_test):,}")

## 2. Initialize SHAP TreeExplainer
`shap.TreeExplainer` is optimized for tree ensembles like Random Forest, calculating exact Shapley values in polynomial time.

In [ ]:
# Subsample 250 test instances (balanced: 50 fraud, 200 normal) for rapid, comprehensive SHAP analysis
fraud_sample = X_test[y_test == 1].head(50)
normal_sample = X_test[y_test == 0].head(200)
sample_X = pd.concat([fraud_sample, normal_sample])
sample_y = pd.concat([y_test[y_test == 1].head(50), y_test[y_test == 0].head(200)])

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample_X)

# Depending on shap / sklearn version, shap_values can be a list [class_0, class_1] or an ndarray
if isinstance(shap_values, list) and len(shap_values) > 1:
    shap_fraud = shap_values[1]
elif hasattr(shap_values, 'shape') and len(shap_values.shape) == 3:
    shap_fraud = shap_values[:, :, 1]
else:
    shap_fraud = shap_values

print("SHAP values computed successfully!")
print(f"Evaluated matrix shape: {shap_fraud.shape}")

## 3. Global Feature Importance (Macro Drivers)
We plot the mean absolute SHAP value for each feature to see the top features driving fraud classifications.

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_fraud, sample_X, plot_type="bar", show=False, max_display=12)
plt.title("Global Feature Importance: Mean |SHAP Value| (Fraud Class)", fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig("../outputs/figures/shap_global_bar.png", dpi=300)
plt.show()
print("Global summary bar plot saved to outputs/figures/shap_global_bar.png")

## 4. SHAP Beeswarm Plot: Feature Impact Directionality
The beeswarm plot reveals how the feature value (low vs high) influences the fraud likelihood.
For instance, strong negative values of `V14`, `V10`, and `V12` correlate with high fraud predictions.

In [ ]:
plt.figure(figsize=(12, 7))
shap.summary_plot(shap_fraud, sample_X, show=False, max_display=12)
plt.title("SHAP Summary Beeswarm: Directional Impact on Fraud Prediction", fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig("../outputs/figures/shap_beeswarm.png", dpi=300)
plt.show()
print("Beeswarm plot saved to outputs/figures/shap_beeswarm.png")

## 5. Local Transaction Explanation: Case Study
We inspect an individual fraudulent transaction to generate a transparent audit breakdown.

In [ ]:
# Select the first confirmed fraud transaction in our sample
fraud_idx = 0
single_shap = shap_fraud[fraud_idx]
single_features = sample_X.iloc[fraud_idx]

impact_df = pd.DataFrame({
    "Feature": sample_X.columns,
    "Feature_Value": single_features.values,
    "SHAP_Contribution": single_shap,
    "Abs_Impact": np.abs(single_shap)
}).sort_values("Abs_Impact", ascending=False)

print("Top 6 Contributing Features for Case Study:")
print(impact_df.head(6))

# Waterfall plot of the top contributors
top6 = impact_df.head(8).sort_values("SHAP_Contribution", ascending=True)
colors = ['#ef4444' if x > 0 else '#3b82f6' for x in top6["SHAP_Contribution"]]

plt.figure(figsize=(10, 5))
plt.barh(top6["Feature"], top6["SHAP_Contribution"], color=colors)
plt.axvline(0, color='#64748b', linestyle='--')
plt.title("Local Transaction Explanation: Feature Risk Contributions", fontsize=13, fontweight='bold')
plt.xlabel("SHAP Value (Red = Pushes Toward Fraud, Blue = Pushes Toward Legitimate)")
plt.tight_layout()
plt.savefig("../outputs/figures/shap_local_waterfall.png", dpi=300)
plt.show()